# Tutorial 2: Writing Custom Features

EEGDash ships with a built-in feature bank, but the real power of the library is that you can add your own features with almost no boilerplate. A custom feature is just a plain Python/NumPy function with two lightweight decorators — no subclassing, no configuration files, no registration steps required.

By the end of this tutorial you will have implemented five features from scratch, validated two of them against the built-in feature bank, and run the full pipeline on a mock dataset.

In [ ]:
import numpy as np
from eegdash.features import (
    feature_predecessor,
    univariate_feature,
    FeatureExtractor,
    extract_features,
)

## 1. The Decorator System

Every EEGDash feature function needs two decorators stacked in a specific order:

- **`@feature_predecessor()`** — tells the pipeline what form the input data has. With no argument it means "I expect the raw signal". You can also pass a preprocessor function to say "I expect the output of *that* function".
- **`@univariate_feature`** — tells the pipeline that your function returns one scalar per channel, i.e. it collapses the time axis and produces an output of shape `(batch, channels)`.

**Decorator order matters:** `@feature_predecessor` must be the *outermost* decorator (listed first), and `@univariate_feature` must be listed second. The data shape contract is: input `x` has shape `(batch, channels, time)`, and a univariate feature should return `(batch, channels)` by reducing the last axis.

In [ ]:
# Minimal "hello world" feature — computes the channel-wise mean
@feature_predecessor()   # outermost: declare input type (raw signal)
@univariate_feature      # innermost: one value per channel
def my_mean(x):
    """Mean amplitude per channel."""
    return x.mean(axis=-1)  # (batch, channels, time) → (batch, channels)

In [ ]:
# Quick sanity check on a toy batch
x_toy = np.random.randn(2, 4, 256)  # 2 windows, 4 channels, 256 samples
out = my_mean(x_toy)
print(f"Input shape:  {x_toy.shape}")
print(f"Output shape: {out.shape}")  # should be (2, 4)

## 2. Implementing Signal Features

Now let's implement three practically useful features that all operate on the raw time-series signal. Each one collapses the time axis (`axis=-1`) and returns one value per channel. Pay attention to how `np.diff` and `np.sign` are used — the key is always operating on the last axis so the function works correctly regardless of the batch size or channel count.

In [ ]:
# Feature 1: Zero crossings
# Counts how many times the signal changes sign — a rough measure of frequency content.
@feature_predecessor()
@univariate_feature
def my_zero_crossings(x):
    """Number of zero crossings per channel."""
    return np.sum(np.diff(np.sign(x), axis=-1) != 0, axis=-1)

In [ ]:
# Feature 2: Line length
# Sum of absolute first differences — measures how "wiggly" the signal is.
# High line length indicates high-frequency or high-amplitude activity.
@feature_predecessor()
@univariate_feature
def my_line_length(x):
    """Cumulative arc length of the signal per channel."""
    return np.sum(np.abs(np.diff(x, axis=-1)), axis=-1)

In [ ]:
# Feature 3: Mean absolute deviation
# Average distance of each sample from the channel mean.
# More robust than variance because it uses absolute values instead of squares.
@feature_predecessor()
@univariate_feature
def my_mean_absolute_deviation(x):
    """Mean absolute deviation per channel."""
    return np.mean(np.abs(x - x.mean(axis=-1, keepdims=True)), axis=-1)

In [ ]:
# Test all three on the same toy batch
x_toy = np.random.randn(2, 4, 256)

zc  = my_zero_crossings(x_toy)
ll  = my_line_length(x_toy)
mad = my_mean_absolute_deviation(x_toy)

print(f"Zero crossings shape:         {zc.shape}")   # (2, 4)
print(f"Line length shape:            {ll.shape}")   # (2, 4)
print(f"Mean abs deviation shape:     {mad.shape}")  # (2, 4)

# Spot-check values look reasonable
print(f"\nZero crossings sample values: {zc[0]}")
print(f"Line length sample values:    {ll[0].round(2)}")
print(f"MAD sample values:            {mad[0].round(4)}")

## 3. Validating Against the Feature Bank

EEGDash includes a built-in feature bank with vetted, optimised implementations of common features. Before using a custom implementation in production it is good practice to verify it produces exactly the same output as the reference. Here we test `my_zero_crossings` and `my_line_length` against their built-in equivalents on a controlled signal where the correct answer is easy to reason about.

In [ ]:
from eegdash.features import signal_zero_crossings, signal_line_length

In [ ]:
# Controlled signal: a sine wave at 10 Hz, sampled at 256 Hz, 1 second long
# At 10 Hz we expect ~20 zero crossings per second (two per cycle).
sfreq = 256
t = np.linspace(0, 1, sfreq, endpoint=False)
sine = np.sin(2 * np.pi * 10 * t)  # shape (256,)

# Broadcast to a batch of shape (2, 4, 256) — same signal in every window/channel
x_sine = np.tile(sine, (2, 4, 1))  # (2, 4, 256)
print(f"Test batch shape: {x_sine.shape}")

In [ ]:
# Validate zero crossings
my_zc_out  = my_zero_crossings(x_sine)
ref_zc_out = signal_zero_crossings(x_sine)

np.testing.assert_array_almost_equal(my_zc_out, ref_zc_out)
print(f"Zero crossings match!  Values per channel: {my_zc_out[0]}")

In [ ]:
# Validate line length
my_ll_out  = my_line_length(x_sine)
ref_ll_out = signal_line_length(x_sine)

np.testing.assert_array_almost_equal(my_ll_out, ref_ll_out)
print(f"Line length match!  Values per channel: {my_ll_out[0].round(4)}")

Both assertions pass without error, which confirms our implementations are numerically identical to the feature bank. `my_mean_absolute_deviation` and `my_spectral_mean_frequency` (introduced in the next section) are genuinely new features not present in the built-in bank, so there is no reference to compare against — which is exactly why the decorator system exists: EEGDash makes it easy to add novel features.

## 4. Custom Preprocessor and Spectral Feature

Some features are most naturally computed on a *transformed* version of the signal rather than the raw time-series. EEGDash handles this with the concept of a **preprocessor**: a function decorated with only `@feature_predecessor()` that takes the raw signal and returns an intermediate representation. Feature functions that depend on that preprocessor declare the dependency by passing it to `@feature_predecessor(my_preprocessor)`.

The preprocessor receives an extra keyword argument `_metadata` from the pipeline, which contains recording metadata such as the sampling frequency. This avoids hard-coding constants and makes the feature portable across datasets.

In [ ]:
from scipy.signal import welch

# Preprocessor: raw signal → power spectral density via Welch's method
# Note: only @feature_predecessor() here — NOT @univariate_feature.
# The output is a tuple (frequencies, power) that downstream features consume.
@feature_predecessor()
def my_spectral_preprocessor(x, /, *, _metadata):
    """Compute Welch PSD for each window and channel.

    Returns
    -------
    f : ndarray, shape (n_freqs,)
    p : ndarray, shape (batch, channels, n_freqs)
    """
    sfreq = _metadata["info"]["sfreq"]
    f, p = welch(x, fs=sfreq, axis=-1)
    return f, p

In [ ]:
# Feature: spectral mean frequency
# The weighted average frequency (centroid of the power spectrum).
# Depends on my_spectral_preprocessor — note the argument to @feature_predecessor.
@feature_predecessor(my_spectral_preprocessor)  # declare dependency
@univariate_feature
def my_spectral_mean_frequency(f, p, /):
    """Power-weighted mean frequency per channel."""
    return np.sum(f * p, axis=-1) / np.sum(p, axis=-1)

In [ ]:
# Test the preprocessor directly
x_toy = np.random.randn(2, 4, 256).astype(np.float32)
f, p = my_spectral_preprocessor(x_toy, _metadata={"info": {"sfreq": 256}})
print(f"Frequency bins: {len(f)}")
print(f"Power shape:    {p.shape}")  # (2, 4, n_freqs)

In [ ]:
# Test the spectral feature
mf = my_spectral_mean_frequency(f, p)
print(f"Mean frequency shape: {mf.shape}")  # (2, 4)
print(f"Mean frequency values (Hz): {mf[0].round(2)}")

## 5. Putting It All Together — FeatureExtractor

`FeatureExtractor` is the object that ties features into a reusable, composable pipeline. You can nest `FeatureExtractor` instances to group related features — for example, one group for raw-signal features and a separate group for spectral features. When a group needs a preprocessor, pass it as the `preprocessor` keyword argument. The pipeline takes care of running each preprocessor exactly once per batch, regardless of how many features depend on it.

In [ ]:
# Build a nested extractor: signal features + spectral features
extractor = FeatureExtractor({
    "signal": FeatureExtractor({
        "zero_crossings": my_zero_crossings,
        "line_length":    my_line_length,
        "mad":            my_mean_absolute_deviation,
    }),
    "spectral": FeatureExtractor(
        {"mean_freq": my_spectral_mean_frequency},
        preprocessor=my_spectral_preprocessor,
    ),
})

In [ ]:
# Build a mock dataset using braindecode utilities
# We create 3 fake recordings, each with 32 windows of 8 channels and 256 samples.
import pandas as pd
from braindecode.datasets import create_from_X_y, BaseConcatDataset

n_recordings = 3
datasets = []
for i in range(n_recordings):
    X = np.random.randn(32, 8, 256).astype(np.float32)
    y = np.zeros(32, dtype=np.int64)
    ds = create_from_X_y(
        X, y,
        drop_last_window=False,
        sfreq=256,
        ch_names=[f"ch{j}" for j in range(8)],
    )
    datasets.append(ds)

mock_ds = BaseConcatDataset(datasets)
print(f"Mock dataset: {n_recordings} recordings, "
      f"{len(mock_ds)} total windows")

In [ ]:
# Run the full feature extraction pipeline
features_ds = extract_features(mock_ds, extractor, batch_size=32, n_jobs=1)

# Inspect the output for the first recording
first_recording = features_ds.datasets[0]
print(f"Feature matrix shape: {first_recording.features.shape}")
print(f"\nFeature columns ({len(first_recording.features.columns)} total):")
for col in first_recording.features.columns:
    print(f"  {col}")

Each row in `features.features` corresponds to one EEG window, and each column corresponds to one (feature, channel) pair. Because we have 8 channels and 4 scalar features (zero_crossings, line_length, mad, mean_freq), the matrix has 32 rows × 32 columns.

## 6. Summary and Next Steps

Here is the complete decorator contract to remember:

| Goal | Decorators (outermost → innermost) |
|------|-------------------------------------|
| Univariate feature on raw signal | `@feature_predecessor()`, `@univariate_feature` |
| Univariate feature on preprocessor output | `@feature_predecessor(my_preprocessor)`, `@univariate_feature` |
| Preprocessor (raw → intermediate) | `@feature_predecessor()` only |

Two of the features built in this tutorial — `my_mean_absolute_deviation` and `my_spectral_mean_frequency` — are **not** in the built-in feature bank, which illustrates why the extension system exists: researchers often need domain-specific measures that no general-purpose library can anticipate.

**Tutorial 3** covers advanced usage: multivariate features (output shape `(batch, n_output_features)` instead of `(batch, channels)`), chaining preprocessors, and parallelising extraction across CPU cores with `n_jobs > 1`.